# Context overview - LangChain

[link](https://docs.langchain.com/oss/python/concepts/context)

## Types of context

Categorized in two dimensions:

- Mutability:
    - Static: doesn't change along single run
        - Examples: user metadata (e.g., profile), database connection details, tools.
    - Dynamic: it changes
        - Examples: conversation history, intermediate results, tool results

- Lifetime:
    - Single run: lost after run.
    - Multiple runs: persists across sessions / conversations


## Context management in LangGraph

| Context type | Description | Mutability | Lifetime | Access method |
|---|---|---|---|---|
| Static runtime context | User metadata, database connections, tools | Static | Single run | `context` argument to `invoke` / `stream` |
| Dynamic runtime context (state) | Conversation history, intermediate results | Dynamic | Single run | LangGraph state object |
| Dynamic cross-conversation context (store) | Persistent data across sessions | Dynamic | Multiple runs | LangGraph store |


## Static context

- Accessed through the `runtime.context` property.
- Set when invoking, `context_schema=MyContextSchemaClass`
- Can be accessed in:
    - Function with `@dynamic_prompt` decorator.
        - This function is provided:
            - In `graph.invoke`, through the `context` argument, as `context={"my_property": "my_value", ...}`
            - In `agent.invoke`, through the `middleware` argument when invoking graph, as `middleware=[function]`.
    - Node, via especial argument.
    - Tool, via especial argument.


## Using generic graph

```python
from dataclasses import dataclass

@dataclass
class ContextSchema:
    user_name: str

graph.invoke (
    {"messages": [{"role": "user", "content": "Hi!"}]},
    context={"user_name": "Jaume"},
)
```


## Using agent

```python
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def personalized_prompt (request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name
    return f"You are a helpful assistant. Address user as {user_name}."

agent = create_agent (
    model="gpt...",
    tools=[get_weather],
    middleware=[personalize_prompt],
    context_schema=ContextSchema
)

agent.invoke (
    {"messages": [{"role": "user", "content": "Hi!"}]},
    context=ContextSchema("Jaume")
)
```

## Using node

```python
from langgraph.runtime import Runtime

def node (state: State, runtime: Runtime[ContextSchema]):
    user_name = runtime.context.user_name
    ...
```

## Using tool

```python
from langchain.tools import ToolRuntime

def my_tool (runtime: ToolRuntime[ContextSchema]):
    user_name = runtime.context.user_name
```

## Dynamic context

- Accessed through:
    - In agent: `request.state.get`, in function with `@dynamic_prompt` decorator.
        - Pass `state_schema=MyStateSchema` to `invoke` call.
        - Again,  pass `middleware=[decorated_function]` in `invoke` call.
    - In node, through the state argument.

## Dynamic context in agent

```python
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.agents import AgentState

class StateSchema (AgentState):
    extra_field: int

@dynamic_prompt
def personalized_prompt (request: ModelRequest) -> str:
    extra_field = request.state.get ("extra_field", 0)
    extra_field += 1
    # request.state["extra_field"] = extra_field ## Is this correct?
    ...

agent = create_agent (
    model="...",
    tools=[...],
    middleware=[personalized_prompt],
    state_schema=StateSchema,
)

agent.invoke (
    {
        "messages": "hi",
        "extra_field": 1
    }
)
```

## Dynamic context, node

```python

from typing import TypedDict
from langchain.messages import AnyMessage

class CustomState (TypedDict):
    messages: list[AnyMessage]
    extra_field: int

def node (state: CustomState):
    extra_field = state["extra_field"]
    return {"extra_field": extra_field + 1}


```

## Dynamic cross-conversation context

Based on store object, see long-term memory

# Context engineering - LangChain

[link](https://docs.langchain.com/oss/python/langchain/context-engineering)

## Overview: Context engineering

Providing the right information and tools in the right format to have the LLM accomplish its task.


## Overview: the agent loop

```mermaid
flowchart
    Request --> Model 
    Model   -- "Action"      --> Tools
    Tools   -- "Observations" --> Model
    Model   --> Result

    linkStyle 1,2 stroke-width:2px,stroke-dasharray:5 5;

    style Model fill:#FFD966,stroke:#333,stroke-width:2px,color:#000
    style Tools fill:#FFD966,stroke:#333,stroke-width:2px,color:#000
    style Request fill:none,stroke:#333,stroke-width:2px,color:#FFFFFF
    style Result fill:none,stroke:#333,stroke-width:2px,color:#FFFFFF
```

- Model: receives prompt and list of tools to be used. It either returns a response or requests the tools to perform actions.
- Tools execution: runs the actions and returns resulting observations to model.


## What you can control

| Context Type | What you control | Transient or Persistent |
| --- | --- | --- |
| Model Context | What goes into model calls (instructions, tools, message history, response format) | Transient |
| Tools Context | What tools can access and produce (reads/writes to state, store, and runtime context) | Persistent |
| Life-cycle Context | What happens between model and tool calls (summarization, guardrails, logging, etc.) | Persistent |


## Data Sources

| Data Source | Also Known As | Scope | Examples |
| --- | --- | --- | --- |
| Runtime Context | Static Configuration | Single conversation | User ID, database connection, API keys, environment settings |
| State | Short-term memory | Single conversation | History of messages, tool results, uploaded files, authentication status |
| Store | Long-term memory | Cross conversation | User preferences, extracted insights, memories, historical data |


## How it works

middleware. It allows to:
- hook into any step of the lifecycle
- update context
- jump to a different step in the lifecycle

## System prompt: state

Example: 
- Depending on history length, we make the response concise or not

```python
from langchain.agents.middleware import ModelRequest, dynamic_prompt

@dynamic_prompt
def f (request: ModelRequest) -> str:
    if len(request.messages) > 10:
        m = "Be concise"
    ...
    return m

create_agent (
    model=...,
    tools=[...],
    middleware=[f]
)
```

## System prompt: store

Example: 
- Depending on user preferences, use one communication style or another

```python
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from dataclasses import dataclass

@dataclass
class ContextSchema:
    user_id: str

@dynamic_prompt
def f (request: ModelRequest) -> str:
    user_id = request.runtime.context.user_id
    store = request.runtime.store
    ...
    return m

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextScheme,
    store=InMemoryStore()
)
```

## System prompt: runtime context

Example: 
- Depending on user priviligies, we request the model to only use read tools or read/write

```python
from langchain.agents.middleware import ModelRequest, dynamic_prompt

@dataclass
class ContextSchema:
    role: str = Field (description="user role: admin, user, guest")

@dynamic_prompt
def f (request: ModelRequest) -> str:
    if request.runtime.context.role == "user":
        m = "Use only read tools"
    ...
    return m

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextSchema,
)
```

## Messages: state

Example: 
- Inject uploaded file context from State when relevant to query.

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    if request.state.get("uploaded_files"):
        file_context = integrate (request.state.get("uploaded_files"))
        messages = [
            *request.messages,
            {"role": "user", "content": file_context}
        ]
    request = request.override (messages=messages)
    ...
    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f]
)
```

## Messages: store

Example: 
- Retrieve user's email writing style from store to guide the model.

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from dataclasses import dataclass

@dataclass
class ContextSchema:
    user_id: str

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    user_id = request.runtime.context.user_id
    store = request.runtime.store
    ...
    new_instructions = integrate (guiding_style)
    messages = [
        *request.messages,
        {"role": "user", "content": new_instructions}
    ]
    request = request.override (messages=messages)

    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextScheme,
    store=InMemoryStore()
)
```

## Messages: runtime context

Example: 
- Depending on user profile details, we request the model certain compliance rules

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from dataclasses import dataclass

@dataclass
class ContextSchema:
    user_id: str

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    profile_details_1 = request.runtime.context.profile_details_1
    ...
    new_instructions = get_rules (profile_details_1, ...)
    messages = [
        *request.messages,
        {"role": "user", "content": new_instructions}
    ]
    request = request.override (messages=messages)

    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextScheme,
)
```

## Tools: annotation

```python
@tool
def my_tool (
    param1: param_type,
    ...
) -> return_type:
    """
        This tool does X and Y based on param1.... Use this tool in this and that situation.

        Args:
            param1: description: value_1, value_2, value_3
    """

```

## Tools: state

Example: 
- Depending on messages or authentication status, restrict usage of tools

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    if request.state.get("authentication")=="Bad":
        tools_to_use = restrict_tools (request.tools, startswith="public")
        
    request = request.override (tools=tools_to_use)
    ...
    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f]
)
```

## Tools: store

Example: 
- Depending on user preferences, use certain tools

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call

@dataclass
class ContextSchema:
    user_id: str

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    preferences = get_preferences (request.runtime.context.user_id, request.runtime.store)
    tools_to_use = restrict_tools (request.tools, preferences)
    request = request.override (tools=tools_to_use)
    ...
    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextSchema,
    store=InMemoryStore()
)
```

## Tools: runtime context

Example: 
- Depending on user details, use certain tools

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call

@dataclass
class ContextSchema:
    user_id: str

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    details = get_details (request.runtime.context.user_details)
    tools_to_use = restrict_tools (request.tools, details)
    request = request.override (tools=tools_to_use)
    ...
    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextSchema,
)
```

## Model: state

Example: 
- Depending on history length, we use a faster or slower model

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.chat_models import init_chat_model

models = {
    "fast": init_chat_model ("gpt..."),
    "slow": init_chat_model ("gpt...")
}

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    
    model = select_model (
        request.state.get("field"), 
        request.messages, 
        models
    )
    request = request.override (model=model)
    
    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f]
)
```

## Model: store

Example: 
- Depending on user preferences, use cheaper or more expensive model

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.chat_models import init_chat_model

models = {
    "fast": init_chat_model ("gpt..."),
    "slow": init_chat_model ("gpt...")
}

@dataclass
class ContextSchema:
    user_id: str

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    preferences = get_preferences (request.runtime.context.user_id, request.runtime.store)
    model = select_model (preferences, models)
    request = request.override (model=model)

    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextSchema,
    store=InMemoryStore()
)

## Model: runtime context

Example: 
- Depending on user and environment details, use certain models

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.chat_models import init_chat_model

models = {
    "fast": init_chat_model ("gpt..."),
    "slow": init_chat_model ("gpt...")
}

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    details = get_details (request.runtime.context.user_details)
    model = select_model (preferences, models)
    request = request.override (model=model)
    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextSchema,
)
```

## Response format: state

Example: 
- Depending on history length and other field values of current state, we use more or less detailed format

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from pydantic import BaseModel

class DetailedFormat (BaseModel):
    """Structured information to be used for this and other purpose"""
    field_1: str = Field (description="...")
    more_details: str = Field (description="...")

class ShortFormat (BaseModel):
    field_1: str = Field (description="...")

formats = {
    "detailed": DetailedFormat,
    "short": ShortFormat
}

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    
    format = select_format (
        request.state.get("field"), 
        request.messages, 
        formats
    )
    request = request.override (response_format=format)
    
    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f]
)
```

## Response format: store

Example: 
- Depending on user preferences, we use more or less detailed format

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from pydantic import BaseModel

class DetailedFormat (BaseModel):
    """Structured information to be used for this and other purpose"""
    field_1: str = Field (description="...")
    more_details: str = Field (description="...")

class ShortFormat (BaseModel):
    field_1: str = Field (description="...")

formats = {
    "detailed": DetailedFormat,
    "short": ShortFormat
}

@dataclass
class ContextSchema:
    user_id: str


@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    preferences = get_preferences (request.runtime.context.user_id, request.runtime.store)
    format = select_format (
        preferences
        formats
    )
    request = request.override (response_format=format)

    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextSchema,
    store=InMemoryStore()
)

## Response format: runtime context

Example: 
- Depending on user and environment details, we use more or less detailed format

```python
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from pydantic import BaseModel

class DetailedFormat (BaseModel):
    """Structured information to be used for this and other purpose"""
    field_1: str = Field (description="...")
    more_details: str = Field (description="...")

class ShortFormat (BaseModel):
    field_1: str = Field (description="...")

formats = {
    "detailed": DetailedFormat,
    "short": ShortFormat
}

@dataclass
class ContextSchema:
    user_details: str

@wrap_model_call
def f (
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    
    details = get_details (request.runtime.context.user_details)
    format = select_format (
        details
        formats
    )
    request = request.override (response_format=format)
    return handler(request)

create_agent (
    model=...,
    tools=[...],
    middleware=[f],
    context_scheme=ContextSchema,
)
```

## Tool Context

## Tool Context: state

Example: 
- Depending on history length and other field values of current state, the tool performs one action or another

```python
from langchain.tools import tool, ToolRuntime

@tool
def my_tool (
    runtime: ToolRuntime
):
    result = obtain_result (
        runtime.state.get("field_1"),
        # request.messages ? 
    )
    return result

create_agent (
    model=...,
    tools=[my_tool],
)
```

## Tool Context: store

Example: 
- Depending on user preferences, the tool performs one action or another

```python
from langchain.tools import tool, ToolRuntime

@dataclass
class ContextSchema:
    user_id: str

@tool
def my_tool (
    runtime: ToolRuntime[ContextSchema]
):
    preferences = get_preferences (runtime.context.user_id, runtime.store)
    result = obtain_result (preferences)
    return result

create_agent (
    model=...,
    tools=[my_tool],
    context_scheme=ContextSchema,
    store=InMemoryStore()
)

## Tool context: runtime context

Example: 
- Depending on user and environment details, the tool performs one action or another

```python
from langchain.tools import tool, ToolRuntime

@dataclass
class ContextSchema:
    user_id: str

@tool
def my_tool (
    runtime: ToolRuntime[ContextSchema]
):
    details = get_details (runtime.context.user_details)
    result = obtain_result (details)
    return result

agent = create_agent (
    model=...,
    tools=[my_tool],
    context_scheme=ContextSchema
)

agent.invoke (
    {"messages": [{"role": "user", "content": "my message"}]},
    context=ContextSchema(
        "user_id": ...
    )
)
```

## Tool Context: state (write)

Example: 
- Depending on history length and other field values of current state, the tool changes the state in certain way

```python
from langchain.tools import tool, ToolRuntime
from langchain.types import Command

@tool
def my_tool (
    runtime: ToolRuntime
) -> Command:
    my_update = obtain_result (
        runtime.state.get("field_1"),
        # request.messages ? 
    )
    return Command (
        update={"my_field": my_update}
    )

create_agent (
    model=...,
    tools=[my_tool],
)
```

## Lifecycle context: hooks

```mermaid
flowchart
    Request --> before_agent
    before_agent --> before_model
    before_model --> Model
    Model["wrap_model_call<br/>Model"] --> after_model
    after_model -.-> Tool
    after_model -.-> after_agent
    Tool["wrap_tool_call<br/>Tool"] --> before_model
    after_agent --> Response
```

## Lifcycle context: actions

- Control what happens between the core agent steps 
    - Good for: intercepting data flow to implement cross-cutting concerns like summarization, guardrails, and logging.

- Update context 
    - Modify state and store to persist changes, update conversation history, or save insights
    - Jump in the lifecycle - Move to different steps in the agent cycle based on context (e.g., skip tool execution if a condition is met, repeat model call with modified context)

## Built-in middleware

Summarization:

```python
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger={"tokens": 4000},
            keep={"messages": 20},
        ),
    ],
)
```

## Best Practices

- Start simple - Begin with static prompts and tools, add dynamics only when needed
- Test incrementally - Add one context engineering feature at a time
- Monitor performance - Track model calls, token usage, and latency
- Use built-in middleware - Leverage SummarizationMiddleware, LLMToolSelectorMiddleware, etc.
- Document your context strategy - Make it clear what context is being passed and why
- Understand transient vs persistent: Model context changes are transient (per-call), while life-cycle context changes persist to state

## Further reading

- [Context Rot: When Long Context Fails](https://www.youtube.com/watch?v=3s_N60u0jEY)
- [Continual In-Context Learning](https://blog.langchain.dev/dosu-langsmith-no-prompt-eng/)
- [Prompt Engineering vs Context Engineering | MIM Technovate](https://www.youtube.com/watch?v=QZFzYMTfxtE&t=320)
- [Prompt Engineering vs Context Engineering | IBM Technology](https://www.youtube.com/watch?v=vD0E3EUb8-8)
- [Context is all you need | Prompt Engineering](https://www.youtube.com/watch?v=ioOHXt7wjhM&t=49)